# AutoGrad

In [5]:
import torch
import math

### Direct Math differentiation

In [6]:
def dy_dx(x):
    return 2*x

In [7]:
dy_dx(3)

6

In [12]:
def dz_dx(x):
    return 2*x*math.cos(x)

In [13]:
dz_dx(3)

-5.939954979602673

### Same using Pytorch

In [15]:
x = torch.tensor(3.0,requires_grad=True)
x

tensor(3., requires_grad=True)

In [16]:
y = x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [17]:
y.backward() # carries out backward pass in y ,i.e dy/dx

In [18]:
x.grad

tensor(6.)

In [ ]:
# z = math.sin(y)
# z

0.4121184852417566

In [44]:
x = torch.tensor(14.0,requires_grad=True)

In [45]:
y = x**2
y

tensor(196., grad_fn=<PowBackward0>)

In [46]:
z = torch.sin(y)
z

tensor(0.9395, grad_fn=<SinBackward0>)

In [47]:
x

tensor(14., requires_grad=True)

In [48]:
y

tensor(196., grad_fn=<PowBackward0>)

In [49]:
z

tensor(0.9395, grad_fn=<SinBackward0>)

In [50]:
z.backward()

In [51]:
x.grad

tensor(9.5891)

In [ ]:
y.grad # Gradients of Leaf tensors are not calculated

/tmp/ipykernel_18792/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


## A Full NN Example

### Using Maths

In [ ]:
x =  torch.tensor(6.7)
y = torch.tensor(0.0)
# Here we wont calculate gradients with respect to x and y, we will calculate wrt to w (weights) and b (bias)

In [57]:
x,y

(tensor(6.7000), tensor(0.))

In [63]:
w = torch.tensor(1.0) #Initial weight
b = torch.tensor(0.0) #Initial bias

In [64]:
def binary_cross_entropy(pred,tar):
    eps = 1e-8
    pred = torch.clamp(pred,eps,1-eps)
    return -(tar*torch.log(pred) + (1-tar)*torch.log(1-pred))

In [65]:
# Forward pass
z = w*x + b # weighted sum (linear part)
y_pred = torch.sigmoid(z)

# Compute loss
loss = binary_cross_entropy(y_pred,y)

In [74]:
y_pred 

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [66]:
loss

tensor(6.7012)

In [67]:
# Manual Derivatives:
# 1. dL/d(y_pred): LOss wrt prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction y_pred wrt to z sigmoid derivative
dy_pred_dz = y_pred*(1-y_pred)

# 3. dz/dw and dz/db: z wrt to w and b
dz_dw = x
dz_db = 1 # bias for our example here contributes directly to z

In [69]:
# Manual Gradient of loss w.r.t weight 
dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_dw

tensor(6.6918)

In [70]:
# Manual Gradient of loss wrt bias
dL_db = dloss_dy_pred * dy_pred_dz * dz_db
dL_db

tensor(0.9988)

### The same using Pytorch

In [ ]:
x =  torch.tensor(6.7)
y = torch.tensor(0.0)
# Here we wont calculate gradients with respect to x and y, we will calculate wrt to w (weights) and b (bias)

In [72]:
w =  torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0,requires_grad=True)

In [73]:
# forward pass
z = w*x + b
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [75]:
loss = binary_cross_entropy(y_pred,y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [ ]:
loss.backward() # calculate the backward pass on loss

In [77]:
w.grad , b.grad

(tensor(6.6918), tensor(0.9988))

### Differentiation of a Multivariate Function

In [78]:
x = torch.tensor([1.0,2.0,3.0],requires_grad=True)

In [80]:
x

tensor([1., 2., 3.], requires_grad=True)

In [81]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [82]:
y.backward()

In [ ]:
x.grad # Here ,grad is an attribute , i.e property

tensor([0.6667, 1.3333, 2.0000])

### Clearing Grad

In [100]:
x =  torch.tensor(2.0,requires_grad=True)
x

tensor(2., requires_grad=True)

In [101]:
y = x**2

y

tensor(4., grad_fn=<PowBackward0>)

In [102]:
y.backward()

In [103]:
x.grad

tensor(4.)

In [104]:
x.grad.zero_()

tensor(0.)

### Disable gradient Tracking

In [105]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [106]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [107]:
y.backward()

In [108]:
x.grad

tensor(4.)

Three ways to stop gradient tracking
- requires_grad_(False) : its used here as an inline attribute
- detach()
- torch.no_grad()

In [109]:
x.requires_grad_(False)

tensor(2.)

In [110]:
x

tensor(2.)

In [111]:
y = x**2
y

tensor(4.)

In [112]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# Using detach() : create a separate tensor , detaching the requirement of gradients
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [115]:
z = x.detach()
z

tensor(2.)

In [116]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [117]:
y1 = z**2
y1

tensor(4.)

In [118]:
y.backward()

In [120]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [121]:
# Using no_grad()
x

tensor(2., requires_grad=True)

In [122]:
with torch.no_grad():
    y = x**2

In [123]:
y

tensor(4.)

In [124]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn